# 🏠 가계 부채 분석

my-hub의 `data.json`을 읽어서 카드 사용액과 총 부채를 계산합니다.
브라우저에서 바로 실행됩니다 (서버 불필요).

In [ ]:
from pyodide.http import pyfetch

resp = await pyfetch("/my-hub/data.json")
data = await resp.json()
print("데이터 로드 완료:", data["meta"].get("updated", ""))

In [ ]:
cards = data["cards"]
total_used = sum(c.get("used") or 0 for c in cards)
total_limit = sum(c["limit"] for c in cards)
affirm_total = sum(a["remaining"] for a in data["affirm"])
total_debt = total_used + affirm_total

print(f"💳 카드 사용액 합계: ${total_used:,.2f}")
print(f"💳 카드 한도 합계:   ${total_limit:,.2f}")
print(f"📊 카드 사용률: {total_used/total_limit*100:.1f}%")
print(f"📦 Affirm 잔액 합계: ${affirm_total:,.2f}")
print(f"💰 총 부채: ${total_debt:,.2f}")

In [ ]:
from collections import defaultdict

by_holder = defaultdict(lambda: [0, 0])
for c in cards:
    by_holder[c["holder"]][0] += c.get("used") or 0
    by_holder[c["holder"]][1] += c["limit"]

print("=== 명의별 카드 사용 현황 ===")
for h, (u, l) in by_holder.items():
    print(f"{h}: ${u:,.2f} / ${l:,.2f} ({u/l*100:.1f}%)")

In [ ]:
goal = data.get("debt_goal", {})
if goal:
    start, target = goal["start"], goal["target"]
    progress = max(0, (start - total_debt) / (start - target) * 100)
    print(f"🎯 부채 감축 목표: ${start:,.2f} → ${target:,.2f}")
    print(f"   현재 부채: ${total_debt:,.2f}")
    print(f"   달성률: {progress:.1f}%")
    print(f"   목표까지: ${total_debt - target:,.2f} 남음")
else:
    print("목표가 설정되어 있지 않습니다.")